# Modules

In [ ]:
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model
from datasets import Dataset

# Model Configuration

In [ ]:
# 1. Load a lightweight model (0.5B is much faster than 1.5B on CPU)
model_id = "Qwen/Qwen2.5-0.5B-Instruct" 
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

# Load Model & Tokenizer

In [ ]:
# Load model in float32 for CPU stability (or bfloat16 if your CPU supports it)
model = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch.float32)
# 2. Apply LoRA (Targeting only essential layers to save CPU cycles)
config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, config)

# Data Preparation

In [ ]:
# 3. Prepare Dataset with the required XML tags
train_df = pd.read_csv("public/train.csv")

def tokenize_function(examples):
    # Construct the full prompt + completion
    texts = [
        f"Question: {q}\n<reasoning>\n{r}\n</reasoning>\n<answer>\n{a}\n</answer>" 
        for q, r, a in zip(examples['input'], examples['reasoning'], examples['answer'])
    ]
    return tokenizer(texts, truncation=True, padding="max_length", max_length=512)

dataset = Dataset.from_pandas(train_df).map(tokenize_function, batched=True)

# Train the Model

In [ ]:
# 4. CPU-Optimized Training Arguments
training_args = TrainingArguments(
    output_dir="./results",
    use_cpu=True,
    # Epoch Settings
    num_train_epochs=3,
    evaluation_strategy="no",
    # Performance Settings
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=1e-4,      
    # Speed Optimizations
    fp16=False,
    bf16=False,
    logging_steps=10,
    save_strategy="no",       # Don't waste time saving checkpoints
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
)

trainer.train()

# Model Inference

In [ ]:
# 5. Inference on Test Set
test_df = pd.read_csv("public/test.csv")
results = []

model.eval()
for _, row in test_df.iterrows():
    prompt = f"Question: {row['input']}\n<reasoning>\n"
    inputs = tokenizer(prompt, return_tensors="pt").to("cpu")
    
    with torch.no_grad():
        output_tokens = model.generate(
            **inputs, 
            max_new_tokens=256,
            do_sample=False, # Greedy search is faster on CPU
            pad_token_id=tokenizer.eos_token_id
        )
    
    full_text = tokenizer.decode(output_tokens[0], skip_special_tokens=True)
    # Ensure the formatting is cleaned up for the submission
    final_output = full_text.replace(f"Question: {row['input']}", "").strip()
    
    results.append({"id": row['id'], "output": final_output})

# Generation Submission

In [ ]:
# 6. Save Submission
pd.DataFrame(results).to_csv("submission.csv", index=False)